In [ ]:
# This is necessary to recognize the modules
import os
import sys
root_path = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(root_path)
from decimal import Decimal
from theOne import theOne
import pandas as pd
import numpy as np
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

## Load the Candles OHLCV

In [ ]:
# Load Candles
clob = CLOBDataSource()
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
TRADING_PAIR = "POL-USDT"
# DAYS = 60

# clob.load_candles_cache(root_path)
# all_candles = clob.get_candles_from_cache(CONNECTOR_NAME, TRADING_PAIR, INTERVALS)
# print(all_candles)

# candles: Candles = all_candles
# candlesdf = candles.data


In [ ]:
# def load_market_data(connector_name: str, TRADING_PAIR: str, data_type: str = "order_book") -> pd.DataFrame:
#     """
#     Load market data from files for a specific connector and trading pair.
    
#     Args:
#         connector_name: Name of the connector (e.g., "bitmart_paper_trade")
#         TRADING_PAIR: Trading pair symbol (e.g., "LINK-USDT")
#         data_type: Type of data to load ("order_book" or "trades")
    
#     Returns:
#         pd.DataFrame: Concatenated DataFrame containing all data from matching files
#     """
#     folder = root_path + f"/data/order_book/"
    
#     # Define the pattern based on data type
#     pattern = "order_book_snapshots" if data_type == "order_book" else "trades"
    
#     # Find all matching files
#     files = [
#         file for file in os.listdir(folder) 
#         if connector_name in file 
#         and TRADING_PAIR in file 
#         and pattern in file
#     ]
    
#     if not files:
#         raise FileNotFoundError(f"No {data_type} files found for {connector_name} {TRADING_PAIR}")
    
#     # Load and concatenate all matching files
#     dfs = []
#     for file in files:
#         df = pd.read_json(folder + "/" + file, lines=True)
#         dfs.append(df)
    
#     return pd.concat(dfs, ignore_index=True)

# # Example usage:
# order_book_df = load_market_data(CONNECTOR_NAME, TRADING_PAIR, "order_book")
# order_book_df.rename(columns={"ts": "timestamp"}, inplace=True)

### Quantise the orderbook data (Fit to each second)

In [ ]:
# # Ensure timestamp columns are of the same type (int)
# candlesdf['timestamp'] = candlesdf['timestamp'].astype(int)
# order_book_df['timestamp'] = order_book_df['timestamp'].astype(int)

# # Merge on 'timestamp'
# candles_and_ob_df = pd.merge(
#     candlesdf,
#     order_book_df,
#     on='timestamp',
#     how='inner',  # Only keep rows with matching timestamps
#     suffixes=('_candle', '_orderbook')
# )

# # Display the merged DataFrame
# candles_and_ob_df['datetime'] = pd.to_datetime(candles_and_ob_df['timestamp'], unit='s')
# candles_and_ob_df

Look for missing seconds

## Add additional collumns

In [ ]:
# candles_and_ob_df['taker_sell_base_volume'] = candles_and_ob_df['volume'] - candles_and_ob_df['taker_buy_base_volume']

# candles_and_ob_df

In [ ]:
from dataHandler import load_candles_and_orderbook

candles_and_ob_df = load_candles_and_orderbook(CONNECTOR_NAME, INTERVALS, TRADING_PAIR)
candles_and_ob_df

### Order fill logic:
- place orders in this format:
- my_bids = [ [Price, Quantity], [Price, Quantity], ...]
- my_asks = [ [Price, Quantity], [Price, Quantity], ...]

(in this case volume above refers to the volume better than my price)

- For each of my bids levels:
  - Full fill of buy order: 
     -  Volume_above + Volume_same + my_bid_level_volume <= taker_sell_base_volume
  - Partial fill of buy:
    - taker_sell_base_volume - Volume_above - Volume_same <= my_bid_level_volume

- For each of my asks levels:
    - Full fill of sell order: 
      - Volume_above + Volume_same + my_ask_level_volume <= taker_buy_base_volume
    - Partial fill of buy:
      - taker_buy_base_volume - Volume_above - Volume_same <= my_ask_level_volume


In [ ]:
# def calculate_fills(candles_and_ob_df, my_bids=None, my_asks=None):
#     """
#     Calculate order fills for each row in candles_and_ob_df for given buy/sell orders.
#     Returns a list of dicts with fill info for each order at each timestamp.
#     """
#     fills = []

#     for idx in range(len(candles_and_ob_df) - 1):  # -1 because we look ahead 1 second
#         row = candles_and_ob_df.iloc[idx]
#         next_row = candles_and_ob_df.iloc[idx + 1]

#         # BUY LIMIT ORDERS
#         if my_bids:
#             for price, qty in my_bids:
#                 # Order book at T₀
#                 bids = row['bids']  # list of [price, volume]
#                 # Calculate V_above and V_same
#                 V_above = sum(v for p, v in bids if p > price)
#                 V_same = sum(v for p, v in bids if p == price)
#                 # S = taker_sell_base_volume at T₁
#                 S = next_row['taker_sell_base_volume']
#                 filled = max(0, min(qty, S - V_above - V_same))
#                 fills.append({
#                     'timestamp': row['timestamp'],
#                     'side': 'buy',
#                     'price': price,
#                     'qty': qty,
#                     'filled': max(0, filled)
#                 })

#         # SELL LIMIT ORDERS
#         if my_asks:
#             for price, qty in my_asks:
#                 asks = row['asks']
#                 V_below = sum(v for p, v in asks if p < price)
#                 V_same = sum(v for p, v in asks if p == price)
#                 B = next_row['taker_buy_base_volume']
#                 filled = max(0, min(qty, B - V_below - V_same))
#                 fills.append({
#                     'timestamp': row['timestamp'],
#                     'side': 'sell',
#                     'price': price,
#                     'qty': qty,
#                     'filled': max(0, filled)
#                 })

#     return fills

# # Example usage:
# my_bids = [[0.2112, 500]]
# my_asks = [[0.2111, 200]]
# fills = calculate_fills(candles_and_ob_df, my_bids=my_bids, my_asks=my_asks)
# fills_df = pd.DataFrame(fills)
# fills_df.columns

# # filtered_df = fills_df[fills_df['filled'] > 0]
# # print(filtered_df[0:20])

Join fills and candles_and_ob_df

In [ ]:
# final_df = pd.merge(
#     candles_and_ob_df,
#     fills_df,
#     on='timestamp',
#     how='left',  # Keep all rows from candles_and_ob_df
#     suffixes=('', '_fill')
# )

# final_df.columns

# Backtest:

In [ ]:
def compute_obp(df, idx, n=5, l=5):
    """
    Compute OBP (Order Book Pressure) at a specific index `idx`
    using n candles and l levels deep.
    """
    start = max(0, idx - n + 1)
    bid_sum = 0
    ask_sum = 0
    
    for i in range(start, idx + 1):
        row = df.iloc[i]
        bids = row['bids'][:l]
        asks = row['asks'][:l]
        bid_sum += sum(qty for price, qty in bids)
        ask_sum += sum(qty for price, qty in asks)

    # no orderbook pressure
    if ask_sum ==0 and bid_sum == 0:
        return 0
    
    if ask_sum == 0:
        return 1  # extremely bullish
    
    return bid_sum / ask_sum


In [ ]:
def li_bid_formula(row, obp_sign, tick_size=0.0001, mu=3):
    """Generate bid orders at multiple levels"""
    best_bid = row['bids'][0][0]
    adjustment = obp_sign * mu * tick_size
    p_base = best_bid + adjustment
    return [
        [p_base, 100],
        [p_base - tick_size, 200],
        [p_base - 2 * tick_size, 400]
    ]

def li_ask_formula(row, obp_sign, tick_size=0.0001, mu=3):
    """Generate ask orders at multiple levels"""
    best_ask = row['asks'][0][0]
    adjustment = obp_sign * mu * tick_size
    p_base = best_ask + adjustment
    return [
        [p_base, 100],
        [p_base + tick_size, 200],
        [p_base + 2 * tick_size, 400]
    ]

In [ ]:
# # the old one without order tracking and refreshing orders
# def calculate_fills_Lietal(candles_and_ob_df, bid_formula=None, ask_formula=None):
#     """
#     Calculate order fills for each row in candles_and_ob_df using dynamic pricing formulas.
#     bid_formula/ask_formula: function(row) -> list of [price, qty]
#     """
#     fills = []
#     tick_size = 0.0001
#     mu = 2
#     n = 5
#     l = 5

#     for idx in range(len(candles_and_ob_df) - 1):
#         row = candles_and_ob_df.iloc[idx]
#         next_row = candles_and_ob_df.iloc[idx + 1]

#         obp = compute_obp(candles_and_ob_df, idx, n=n, l=l)
#         obp_sign = 1 if obp > 1 else (-1 if obp < 1 else 0)

#         bids_to_place = li_bid_formula(row, obp_sign, tick_size, mu)
#         asks_to_place = li_ask_formula(row, obp_sign, tick_size, mu)

#         # Calculate fills like before
#         for price, qty in bids_to_place:
#             V_above = sum(v for p, v in row['bids'] if p > price)
#             V_same = sum(v for p, v in row['bids'] if p == price)
#             V_before = V_above + V_same
#             S = next_row['taker_sell_base_volume']
#             filled = max(0, min(qty, S - V_above - V_same))
#             fills.append({
#                 'timestamp': row['timestamp'],
#                 'V_before': V_before,
#                 'Sell_VOL': S,
#                 'side': 'buy',
#                 'price': price,
#                 'qty': qty,
#                 'filled': max(0, filled)
#             })

#         for price, qty in asks_to_place:
#             V_below = sum(v for p, v in row['asks'] if p < price)
#             V_same = sum(v for p, v in row['asks'] if p == price)
#             V_beforea = V_below + V_same
#             B = next_row['taker_buy_base_volume']
#             filled = max(0, min(qty, B - V_below - V_same))
#             fills.append({
#                 'timestamp': row['timestamp'],
#                 'V_before': V_beforea,
#                 'Buy_VOL': B,
#                 'side': 'sell',
#                 'price': price,
#                 'qty': qty,
#                 'filled': max(0, filled)
#             })
#     return fills
# fills = calculate_fills_Lietal(candles_and_ob_df, bid_formula=li_bid_formula, ask_formula=li_ask_formula)
# fills_df = pd.DataFrame(fills)
# fills_df
# filtered_df = fills_df[(fills_df['filled'] > 0)]
# # filtered_df[0:20]

Questions I have up untill the prev code block:
- say i buy 100 units at time t0 
  - should i make the availible volume at t1 100 less ?
- OR
  - should i rather update my order and the availible buy(taker_sell_base_volume) and sell(taker_buy_base_volume) volume periodically ?

Here is an example of what i'm talking about:
timestamp	V_before	Sell_VOL	side	price	qty	filled	Buy_VOL
45	1748699407	0	NaN	sell	0.211	100	62.3	62.3
46	1748699407	0	NaN	sell	0.2111	200	62.3	62.3
75	1748699412	0	NaN	sell	0.211	100	32.4	32.4
76	1748699412	0	NaN	sell	0.2111	200	32.4	32.4

- the answer is no:
  - Rather do the following:
  - If buy_vol/sell_vol is the same in the next timestamp - Don't fill a new order
  - If buy/ sell order is filled - update the order & refresh order every x amount of iterations(seconds)
  - my orders at level1 should be placed in the order book such that my orders at level2, 3, ... also sees it as part of its V_above or volume (better priced than it)


In [ ]:
candles_and_ob_df.columns

In [ ]:
def calculate_fills_Lietal_refresh_debug(
    candles_and_ob_df, 
    bid_formula=None, 
    ask_formula=None, 
    refresh_interval=5
):
    """
    Calculate order fills for each row in candles_and_ob_df using dynamic pricing formulas.
    - Only fill if taker volume changes at T+1.
    - Refresh orders every `refresh_interval` seconds.
    - Each order level includes own better-priced orders in V_above/V_below.
    - NOW: Records ALL order details for every row for debugging purposes.
    
    Returns a list with one entry per row per order level, showing:
    - All outstanding orders (even if not filled)
    - Fill status and amounts
    - Volume calculations for debugging
    """
    debug_records = []
    tick_size = 0.0001
    mu = 2
    n = 5
    l = 5

    outstanding_bids = []
    outstanding_asks = []
    last_refresh = -refresh_interval

    for idx in range(len(candles_and_ob_df) - 1):
        row = candles_and_ob_df.iloc[idx]
        next_row = candles_and_ob_df.iloc[idx + 1]

        # Check volume changes
        buy_vol_changed = row['taker_buy_base_volume'] != next_row['taker_buy_base_volume']
        sell_vol_changed = row['taker_sell_base_volume'] != next_row['taker_sell_base_volume']
        
        # Calculate incremental volumes
        S = next_row['taker_sell_base_volume'] - row['taker_sell_base_volume'] if sell_vol_changed else 0
        B = next_row['taker_buy_base_volume'] - row['taker_buy_base_volume'] if buy_vol_changed else 0

        # Refresh orders every refresh_interval seconds
        order_refreshed = False
        if (idx - last_refresh) >= refresh_interval:
            obp = compute_obp(candles_and_ob_df, idx, n=n, l=l)
            obp_sign = 1 if obp > 1 else (-1 if obp < 1 else 0)
            outstanding_bids = bid_formula(row, obp_sign, tick_size, mu)
            outstanding_asks = ask_formula(row, obp_sign, tick_size, mu)
            outstanding_bids = [[price, qty] for price, qty in outstanding_bids]
            outstanding_asks = [[price, qty] for price, qty in outstanding_asks]
            last_refresh = idx
            order_refreshed = True

        # Process BUY LIMIT ORDERS (filled when market sells occur)
        if outstanding_bids:
            # Sort bids by price descending for proper fill priority
            outstanding_bids_sorted = sorted(enumerate(outstanding_bids), 
                                           key=lambda x: x[1][0], reverse=True)
            
            remaining_sell_volume = S
            
            for level, (original_idx, (price, qty)) in enumerate(outstanding_bids_sorted):
                # Calculate volumes for this order
                V_above = sum(v for p, v in row['bids'] if p > price)
                own_better = sum(outstanding_bids[j][1] for j in range(len(outstanding_bids)) 
                               if j != original_idx and outstanding_bids[j][0] > price and outstanding_bids[j][1] > 0)
                V_above_total = V_above + own_better
                V_same = sum(v for p, v in row['bids'] if p == price)
                V_before = V_above_total + V_same
                
                # Calculate fill amount
                filled = 0
                if sell_vol_changed and remaining_sell_volume > 0 and qty > 0:
                    available_for_this_order = max(0, remaining_sell_volume - V_above_total - V_same)
                    fillable = max(0, min(qty, available_for_this_order))
                    if fillable > 0:
                        filled = fillable
                        outstanding_bids[original_idx][1] -= fillable
                        remaining_sell_volume -= fillable
                
                # Record debug info for this order level
                debug_records.append({
                    'timestamp': row['timestamp'],
                    'idx': idx,
                    'side': 'buy',
                    'level': level + 1,
                    'price': price,
                    'qty': qty,
                    'filled': filled,
                    'V_before': V_before,
                    'V_above': V_above,
                    'V_same': V_same,
                    'own_better': own_better,
                    'V_above_total': V_above_total,
                    'market_volume': S,
                    'remaining_volume': remaining_sell_volume + filled,  # Before this order filled
                    'volume_changed': sell_vol_changed,
                    'order_refreshed': order_refreshed,
                    'obp': compute_obp(candles_and_ob_df, idx, n=n, l=l) if idx >= n else None,
                    'best_bid': row['bids'][0][0] if row['bids'] else None,
                    'best_ask': row['asks'][0][0] if row['asks'] else None,
                    'spread': (row['asks'][0][0] - row['bids'][0][0]) if (row['bids'] and row['asks']) else None
                })

        # Process SELL LIMIT ORDERS (filled when market buys occur)
        if outstanding_asks:
            # Sort asks by price ascending for proper fill priority
            outstanding_asks_sorted = sorted(enumerate(outstanding_asks), 
                                           key=lambda x: x[1][0])
            
            remaining_buy_volume = B
            
            for level, (original_idx, (price, qty)) in enumerate(outstanding_asks_sorted):
                # Calculate volumes for this order
                V_below = sum(v for p, v in row['asks'] if p < price)
                own_better = sum(outstanding_asks[j][1] for j in range(len(outstanding_asks)) 
                               if j != original_idx and outstanding_asks[j][0] < price and outstanding_asks[j][1] > 0)
                V_below_total = V_below + own_better
                V_same = sum(v for p, v in row['asks'] if p == price)
                V_before = V_below_total + V_same
                
                # Calculate fill amount
                filled = 0
                if buy_vol_changed and remaining_buy_volume > 0 and qty > 0:
                    available_for_this_order = max(0, remaining_buy_volume - V_below_total - V_same)
                    fillable = max(0, min(qty, available_for_this_order))
                    if fillable > 0:
                        filled = fillable
                        outstanding_asks[original_idx][1] -= fillable
                        remaining_buy_volume -= fillable
                
                # Record debug info for this order level
                debug_records.append({
                    'timestamp': row['timestamp'],
                    'idx': idx,
                    'side': 'sell',
                    'level': level + 1,
                    'price': price,
                    'qty': qty,
                    'filled': filled,
                    'V_before': V_before,
                    'V_below': V_below,
                    'V_same': V_same,
                    'own_better': own_better,
                    'V_below_total': V_below_total,
                    'market_buy_volume': B,
                    'remaining_volume': remaining_buy_volume + filled,  # Before this order filled
                    'volume_changed': buy_vol_changed,
                    'order_refreshed': order_refreshed,
                    'obp': compute_obp(candles_and_ob_df, idx, n=n, l=l) if idx >= n else None,
                    'best_bid': row['bids'][0][0] if row['bids'] else None,
                    'best_ask': row['asks'][0][0] if row['asks'] else None,
                    'spread': (row['asks'][0][0] - row['bids'][0][0]) if (row['bids'] and row['asks']) else None
                })

        # Clean up filled orders
        outstanding_bids = [order for order in outstanding_bids if order[1] > 0]
        outstanding_asks = [order for order in outstanding_asks if order[1] > 0]

    return debug_records

def analyze_debug_data(debug_df):
    """
    Analyze the debug data to provide insights
    """
    if debug_df.empty:
        return "No data to analyze"
    
    analysis = {}
    
    # Basic statistics
    analysis['total_rows'] = len(debug_df)
    analysis['unique_timestamps'] = debug_df['timestamp'].nunique()
    analysis['buy_orders'] = len(debug_df[debug_df['side'] == 'buy'])
    analysis['sell_orders'] = len(debug_df[debug_df['side'] == 'sell'])
    
    # Fill statistics
    filled_orders = debug_df[debug_df['filled'] > 0]
    analysis['total_fills'] = len(filled_orders)
    analysis['buy_fills'] = len(filled_orders[filled_orders['side'] == 'buy'])
    analysis['sell_fills'] = len(filled_orders[filled_orders['side'] == 'sell'])
    analysis['fill_rate'] = (analysis['total_fills'] / analysis['total_rows']) * 100 if analysis['total_rows'] > 0 else 0
    
    # Volume statistics
    analysis['total_filled_volume'] = debug_df['filled'].sum()
    analysis['avg_fill_size'] = filled_orders['filled'].mean() if len(filled_orders) > 0 else 0
    analysis['max_fill_size'] = debug_df['filled'].max()
    
    # Market volume statistics
    volume_changed = debug_df[debug_df['volume_changed'] == True]
    analysis['periods_with_volume_change'] = len(volume_changed)
    # analysis['avg_market_volume'] = volume_changed['market_volume'].mean() if len(volume_changed) > 0 else 0
    
    # Order refresh statistics
    refreshed = debug_df[debug_df['order_refreshed'] == True]
    analysis['order_refreshes'] = len(refreshed)
    
    return analysis

def get_fills_only(debug_records):
    """
    Extract only the filled orders from debug records for backward compatibility
    """
    fills = []
    for record in debug_records:
        if record['filled'] > 0:
            fills.append({
                'timestamp': record['timestamp'],
                'side': record['side'],
                'price': record['price'],
                'qty': record['qty'],
                'V_before': record['V_before'],
                'Buy_VOL': record.get('market_volume', 0) if record['side'] == 'sell' else 0,
                'Sell_VOL': record.get('market_volume', 0) if record['side'] == 'buy' else 0,
                'filled': record['filled'],
                'level': record['level']
            })
    return fills


In [ ]:

# Get debug data with all order information
debug_records = calculate_fills_Lietal_refresh_debug(
    candles_and_ob_df, 
    bid_formula=li_bid_formula, 
    ask_formula=li_ask_formula, 
    refresh_interval=5
)

# Convert to DataFrame for analysis
debug_df = pd.DataFrame(debug_records)



## Backtest Data Analysis

In [ ]:
def calculate_portfolio_metrics(debug_df, initial_base_stock=0, initial_quote_stock=10000):
    """
    Calculate and append base stock, quote stock, and PnL columns to debug_df.
    
    Parameters:
    -----------
    debug_df : pd.DataFrame
        Debug dataframe from calculate_fills_Lietal_refresh_debug
    initial_base_stock : float
        Starting amount of base asset (default: 0)
    initial_quote_stock : float  
        Starting amount of quote asset (default: 10000)
    
    Returns:
    --------
    pd.DataFrame
        Original dataframe with added columns:
        - base_stock: Running balance of base asset
        - quote_stock: Running balance of quote asset  
        - trade_pnl: PnL from this specific trade (0 if no fill)
        - cumulative_pnl: Running cumulative PnL
        - total_portfolio_value: Base value + Quote stock at current price
        - unrealized_pnl: Mark-to-market PnL on base holdings
    """
    
    # Create a copy to avoid modifying original
    df = debug_df.copy()
    
    # Sort by timestamp and idx to ensure chronological order
    df = df.sort_values(['timestamp', 'idx', 'side', 'level']).reset_index(drop=True)
    
    # Initialize tracking variables
    base_stock = initial_base_stock
    quote_stock = initial_quote_stock
    cumulative_pnl = 0
    initial_portfolio_value = initial_quote_stock  # Assuming we start with quote asset only
    
    # Initialize new columns
    df['base_stock'] = 0.0
    df['quote_stock'] = 0.0
    df['trade_pnl'] = 0.0
    df['cumulative_pnl'] = 0.0
    df['total_portfolio_value'] = 0.0
    df['unrealized_pnl'] = 0.0
    
    # Track weighted average cost basis for PnL calculations
    avg_cost_basis = 0.0
    
    for i in range(len(df)):
        row = df.iloc[i]
        
        # Calculate trade PnL and update positions if there was a fill
        trade_pnl = 0
        
        if row['filled'] > 0:
            fill_amount = row['filled']
            fill_price = row['price']
            
            if row['side'] == 'buy':
                # Buy order filled: spend quote asset, gain base asset
                quote_spent = fill_amount * fill_price
                quote_stock -= quote_spent
                base_stock += fill_amount
                
                # Update weighted average cost basis
                if base_stock > 0:
                    total_cost = (avg_cost_basis * (base_stock - fill_amount)) + (fill_price * fill_amount)
                    avg_cost_basis = total_cost / base_stock
                
                # For buy trades, PnL is negative (cost of acquisition)
                trade_pnl = -quote_spent
                
            elif row['side'] == 'sell':
                # Sell order filled: gain quote asset, lose base asset
                quote_gained = fill_amount * fill_price
                quote_stock += quote_gained
                base_stock -= fill_amount
                
                # Calculate realized PnL based on cost basis
                if avg_cost_basis > 0:
                    cost_of_sold = fill_amount * avg_cost_basis
                    trade_pnl = quote_gained - cost_of_sold
                else:
                    # If no cost basis (e.g., started with base asset), use fill price as PnL
                    trade_pnl = quote_gained
        
        # Update cumulative PnL
        cumulative_pnl += trade_pnl
        
        # Calculate current portfolio value and unrealized PnL
        current_price = row['best_bid'] if row['best_bid'] is not None else row['price']
        base_value = base_stock * current_price
        total_portfolio_value = base_value + quote_stock
        
        # Unrealized PnL is difference between current value and cost basis of holdings
        if base_stock > 0 and avg_cost_basis > 0:
            unrealized_pnl = base_stock * (current_price - avg_cost_basis)
        else:
            unrealized_pnl = 0
        
        # Update dataframe
        df.at[i, 'base_stock'] = base_stock
        df.at[i, 'quote_stock'] = quote_stock
        df.at[i, 'trade_pnl'] = trade_pnl
        df.at[i, 'cumulative_pnl'] = cumulative_pnl
        df.at[i, 'total_portfolio_value'] = total_portfolio_value
        df.at[i, 'unrealized_pnl'] = unrealized_pnl
    
    return df


def portfolio_summary(df_with_portfolio):
    """
    Generate a summary of portfolio performance
    """
    if df_with_portfolio.empty:
        return "No data to analyze"
    
    # Get final values
    final_row = df_with_portfolio.iloc[-1]
    
    # Get filled trades only
    trades = df_with_portfolio[df_with_portfolio['filled'] > 0]
    
    summary = {
        'total_trades': len(trades),
        'buy_trades': len(trades[trades['side'] == 'buy']),
        'sell_trades': len(trades[trades['side'] == 'sell']),
        'total_base_traded': trades['filled'].sum(),
        'final_base_stock': final_row['base_stock'],
        'final_quote_stock': final_row['quote_stock'],
        'total_portfolio_value': final_row['total_portfolio_value'],
        'cumulative_realized_pnl': final_row['cumulative_pnl'],
        'unrealized_pnl': final_row['unrealized_pnl'],
        'total_pnl': final_row['cumulative_pnl'] + final_row['unrealized_pnl'],
        'profitable_trades': len(trades[trades['trade_pnl'] > 0]),
        'losing_trades': len(trades[trades['trade_pnl'] < 0]),
        'avg_trade_pnl': trades['trade_pnl'].mean() if len(trades) > 0 else 0,
        'max_trade_profit': trades['trade_pnl'].max() if len(trades) > 0 else 0,
        'max_trade_loss': trades['trade_pnl'].min() if len(trades) > 0 else 0
    }
    
    return summary



In [ ]:

# Example usage:
# """
# Add portfolio metrics to your debug dataframe
debug_df_with_portfolio = calculate_portfolio_metrics(
    debug_df, 
    initial_base_stock=5000,    # Start with 0 base asset
    initial_quote_stock=20000  # Start with 10,000 quote asset
)

# Get portfolio summary
summary = portfolio_summary(debug_df_with_portfolio)
for key, value in summary.items():
    print(f"{key}: {value}")

# View specific columns
columns_to_view = ['timestamp', 'side', 'level', 'price', 'filled', 
                  'base_stock', 'quote_stock', 'trade_pnl', 'cumulative_pnl']
print(debug_df_with_portfolio[debug_df_with_portfolio['filled'] > 0][columns_to_view])
# """

TODO: See how CUM PNL is calculated - do a sense check
